# Stage 2 Lazy Frame Dataset Sanity Check

This notebook performs sanity checks on the `VADLazyFrameDataset` wrapper for Stage 2 lazy frame-level features with context stacking.

- Verifies frame-level dataset construction and loading
- Checks frame shapes and dimensions (1331 features)
- Validates context stacking and caching
- Ensures binary labels and finite feature values
- Confirms subset sampling works correctly

In [3]:
from pathlib import Path
import sys
import importlib

# detect project root
current = Path.cwd()
project_root = None
for parent in [current] + list(current.parents):
    if (parent / "src").exists() and (parent / "data").exists():
        project_root = parent
        break

if project_root is None:
    raise RuntimeError("Could not detect project root.")

print("PROJECT_ROOT:", project_root)

# add the stage2 module folder directly
stage2_dir = project_root / "src" / "07_stage2_lazy_pipeline"
if str(stage2_dir) not in sys.path:
    sys.path.insert(0, str(stage2_dir))

# import and force reload
import lazy_frame_dataset
importlib.reload(lazy_frame_dataset)

VADLazyFrameDataset = lazy_frame_dataset.VADLazyFrameDataset

generated_dir = project_root / "data" / "generated"
norm_stats_path = generated_dir / "train" / "features" / "noisy_norm_stats.npz"

print("generated_dir:", generated_dir)
print("norm_stats_path:", norm_stats_path)
print("norm_stats exists:", norm_stats_path.exists())

PROJECT_ROOT: /Users/hongjingren/Documents/6140/Noise-Robust-Voice-Activity-Detection-FINAL
generated_dir: /Users/hongjingren/Documents/6140/Noise-Robust-Voice-Activity-Detection-FINAL/data/generated
norm_stats_path: /Users/hongjingren/Documents/6140/Noise-Robust-Voice-Activity-Detection-FINAL/data/generated/train/features/noisy_norm_stats.npz
norm_stats exists: True


In [4]:
import inspect

print(lazy_frame_dataset.__file__)
print(inspect.getsource(lazy_frame_dataset.VADLazyFrameDataset.__init__))

/Users/hongjingren/Documents/6140/Noise-Robust-Voice-Activity-Detection-FINAL/src/07_stage2_lazy_pipeline/lazy_frame_dataset.py
    def __init__(
        self,
        generated_dir: str | Path,
        split: str,
        manifest_type: str = "noisy",
        norm_stats_path: str | Path | None = None,
        subset_fraction: float | None = None,
        subset_seed: int = 1337,
        context_left: int = 5,
        context_right: int = 5,
    ) -> None:
        super().__init__()

        # Assign attributes early
        self.generated_dir = Path(generated_dir).expanduser().resolve()
        self.split = split
        self.manifest_type = manifest_type
        self.context_left = context_left
        self.context_right = context_right

        # Build the underlying sequence dataset
        self.sequence_dataset = VADLazySequenceDataset(
            generated_dir=generated_dir,
            split=split,
            manifest_type=manifest_type,
            norm_stats_path=norm_stats_

In [5]:
dataset = VADLazyFrameDataset(
    generated_dir=str(generated_dir),
    split="dev",
    manifest_type="noisy",
    norm_stats_path=str(norm_stats_path),
    subset_fraction=0.25,
    subset_seed=1337,
)

print("Dataset length:", len(dataset))

sample = dataset[0]
print("ex_id:", sample["ex_id"])
print("x shape:", sample["x"].shape)
print("y shape:", sample["y"].shape)
print("frame_idx:", sample["frame_idx"])

Loaded 500 examples from dev noisy manifest
Applied subset sampling: 125/500 examples (25.00%)
Frame dataset: 280928 total frames from 125 sequences
Dataset length: 280928


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ex_id: dev_0000002
x shape: torch.Size([1331])
y shape: torch.Size([])
frame_idx: 0


In [8]:
# Inspect a few random samples
import random

if len(dataset) > 5:
    random_indices = random.sample(range(len(dataset)), 5)
    print(f"Inspecting random samples at indices: {random_indices}")
    
    for idx in random_indices:
        sample = dataset[idx]
        print(f"  Index {idx}: ex_id={sample['ex_id']}, frame_idx={sample['frame_idx']}, y={sample['y'].item():.1f}")
        
        # Quick checks
        assert sample['x'].shape == (1331,), f"Shape error at index {idx}"
        assert sample['y'].dim() == 0, f"Scalar error at index {idx}"
        assert 0.0 <= sample['y'].item() <= 1.0, f"Binary label error at index {idx}: {sample['y'].item()}"
    
    print("✓ Random samples look good")
else:
    print("Dataset too small for random inspection")

Inspecting random samples at indices: [124262, 264427, 153321, 42470, 85220]
  Index 124262: ex_id=dev_0000218, frame_idx=818, y=1.0
  Index 264427: ex_id=dev_0000447, frame_idx=769, y=1.0
  Index 153321: ex_id=dev_0000277, frame_idx=252, y=1.0
  Index 42470: ex_id=dev_0000062, frame_idx=1063, y=0.0
  Index 85220: ex_id=dev_0000142, frame_idx=1222, y=1.0
✓ Random samples look good


In [9]:
# Verify finite values and binary labels across more samples
import torch

finite_count = 0
binary_count = 0
total_checked = min(100, len(dataset))  # Check up to 100 samples

for i in range(total_checked):
    sample = dataset[i]
    x = sample['x']
    y = sample['y']
    
    # Check finite values
    if torch.isfinite(x).all():
        finite_count += 1
    
    # Check binary labels (allow small tolerance for float precision)
    if abs(y.item() - 0.0) < 1e-6 or abs(y.item() - 1.0) < 1e-6:
        binary_count += 1

print(f"Checked {total_checked} samples:")
print(f"  Finite x values: {finite_count}/{total_checked}")
print(f"  Binary y values: {binary_count}/{total_checked}")

assert finite_count == total_checked, f"Found non-finite x values in {total_checked - finite_count} samples"
assert binary_count == total_checked, f"Found non-binary y values in {total_checked - binary_count} samples"
print("✓ All checked samples have finite features and binary labels")

Checked 100 samples:
  Finite x values: 100/100
  Binary y values: 100/100
✓ All checked samples have finite features and binary labels


## Summary

- Dataset loads successfully with subset sampling (25% of dev set)
- Frame-level indexing provides 1331-dimensional features with context stacking
- Labels are binary (0.0 or 1.0) and features are finite
- Caching and frame mapping work correctly
- Ready for Stage 2 MLP training experiments